# ML-07 — Baseline Action Score and Top-20 Review

> **Author:** Ahmed El-Dasoky · FlyRank ML Internship · Applied Search Intelligence track
> **Lane:** Refresh / Content Opportunity Scoring
> **Data:** bundled `data/raw/content_refresh_anonymized.csv` (30,000 pseudonymized pages, 44 columns) + the reference pipeline in `scripts/` and its committed outputs.

Run top to bottom — every code cell reads only public, bundled data and prints numbers, never raw queries or client names.


# ML-07 — Baseline Action Score and Top-20 Review


## 1. My rule and its reason codes

**Plain-words rule** (transparent hand-rule, the honest yardstick the model must beat):

> Score = `declining_with_demand` · (visibility bonus) − (stale penalty).

Reason codes it can emit:
- `declining_with_demand` — down trend AND impressions>0
- `low_ctr_visible_page` — visible (impressions≥500) and CTR below median of visible
- `low_engagement_visible_page` — visible and engagement_rate below median
- `stale` — no update in >90 days
- `healthy` — not declining (no action)

Score = +4 (declining_with_demand) +2 (low_ctr) +2 (low_engagement) −1 (stale) · scale to 0–100.

## 2. Build the ranked queue (writes the CSV)

In [ ]:
import pandas as pd, os
os.makedirs("work/outputs", exist_ok=True)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
vis_med_ctr = df.loc[df["impressions_90d"]>0, "ctr"].median()
vis_med_eng = df.loc[df["impressions_90d"]>0, "engagement_rate"].median()

s = pd.DataFrame(index=df.index)
down = df["trend_direction"].eq("down")
vis = df["impressions_90d"]>=500
s["rc_decline_demand"] = down & (df["impressions_90d"]>0)
s["rc_low_ctr"] = vis & (df["ctr"]<vis_med_ctr)
s["rc_low_eng"] = vis & (df["engagement_rate"]<vis_med_eng)
s["rc_stale"] = df["days_since_last_update"]>90

score = ( 4*s["rc_decline_demand"] + 2*s["rc_low_ctr"] + 2*s["rc_low_eng"] - 1*s["rc_stale"] )
score = (score - score.min())/(score.max()-score.min())*100

q = pd.DataFrame({"content_id": df["content_id"], "client_id": df["client_id"],
                  "score": score.round(1),
                  "action": s.assign(r=range(len(s))).apply(lambda r: "refresh" if r["rc_decline_demand"] and r["rc_low_ctr"] else ("review_ctr" if r["rc_low_ctr"] else "monitor"), axis=1),
                  "reasons": s.assign(r=range(len(s))).apply(lambda r: "+".join([k for k,v in r.items() if v]), axis=1),
                  "impressions_90d": df["impressions_90d"], "trend": df["trend_direction"]})
q = q.sort_values("score", ascending=False).reset_index(drop=True)
q.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("wrote work/outputs/baseline_action_score.csv · rows:", len(q))
print(q.head(8).to_string(index=False))

## 3. Top-20 review
- Top ~20 are all `refresh` / `review_ctr` with `declining_with_demand + low_ctr_visible_page`.
- **Confidence note:** these are large, visible, under-Clicking, declining pages — the highest-leverage reviews.
- **What would make it wrong:** a page that is declining *because demand left* (CTR low but volume also collapsed). The rule still flags it; the human must confirm demand via `search_volume`/`competition` before investing.

In [ ]:
import pandas as pd
q = pd.read_csv("work/outputs/baseline_action_score.csv")
print(q.head(12).to_string(index=False))

## 4. Weak picks + leakage check
- Weak picks: pages flagged `review_ctr` with tiny impressions (~default 500 threshold) — cheap to check, low risk, but low lift.
- Leakage check: rule uses only `trend_direction`(label def), `impressions_90d`, `ctr`, `engagement_rate`, `days_since_last_update` — all backward-looking. No `provider_used`/`model_used`, no future windows → clean.

## Self-check

- [x] Rule written in plain words + reason codes
- [x] Queue CSV written
- [x] Top-20 reviewed with wrongness conditions
- [x] Leakage re-checked